In [2]:
#Cell 1 — Imports and configuration
# ============================================================
# NOTEBOOK 10 — SECONDARY MULTICLASS TASK
# ============================================================
# Objective:
# Predict three academic-risk categories:
#
#   0 = Low Risk
#   1 = Moderate Risk
#   2 = High Risk
#
# Locked imbalance strategy:
#   CLASS_WEIGHTING
#
# Primary multiclass metric:
#   F1-Macro
#
# Independent test set:
#   117 observations
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize

from xgboost import XGBClassifier


print("=" * 70)
print("NOTEBOOK 10 — SECONDARY MULTICLASS TASK")
print("=" * 70)

print("✓ Libraries imported successfully")

NOTEBOOK 10 — SECONDARY MULTICLASS TASK
✓ Libraries imported successfully


In [3]:
# ============================================================
# CELL 2 — PROJECT PATHS AND MULTICLASS DATA LOADING
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\HP\Documents\SP-XGBOOST"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print("=" * 70)
print("PROJECT PATH VALIDATION")
print("=" * 70)

print(f"Project directory: {PROJECT_DIR}")
print(f"Processed data:   {PROCESSED_DIR}")
print(f"Models:           {MODELS_DIR}")
print(f"Results:          {RESULTS_DIR}")

# ------------------------------------------------------------
# Validate directories
# ------------------------------------------------------------

assert PROJECT_DIR.exists(), "Project directory not found"
assert PROCESSED_DIR.exists(), "Processed data directory not found"
assert MODELS_DIR.exists(), "Models directory not found"
assert RESULTS_DIR.exists(), "Results directory not found"

print()
print("✓ All project directories validated")

# ------------------------------------------------------------
# Load Notebook 3 multiclass datasets
# ------------------------------------------------------------

train_multiclass_processed = pd.read_csv(
    PROCESSED_DIR / "train_multiclass_processed.csv"
)

test_multiclass_processed = pd.read_csv(
    PROCESSED_DIR / "test_multiclass_processed.csv"
)

print()
print("=" * 70)
print("MULTICLASS DATASETS LOADED")
print("=" * 70)

print(
    f"Training shape: {train_multiclass_processed.shape}"
)

print(
    f"Testing shape:  {test_multiclass_processed.shape}"
)

PROJECT PATH VALIDATION
Project directory: C:\Users\HP\Documents\SP-XGBOOST
Processed data:   C:\Users\HP\Documents\SP-XGBOOST\data\processed
Models:           C:\Users\HP\Documents\SP-XGBOOST\models
Results:          C:\Users\HP\Documents\SP-XGBOOST\results

✓ All project directories validated

MULTICLASS DATASETS LOADED
Training shape: (468, 62)
Testing shape:  (117, 62)


In [4]:
# ============================================================
# CELL 3 — MULTICLASS TARGET / PREDICTOR SEPARATION
# ============================================================

print("=" * 70)
print("MULTICLASS TARGET / PREDICTOR SEPARATION")
print("=" * 70)


MULTICLASS_TARGET = "RISK_LABEL"


# ------------------------------------------------------------
# Validate target
# ------------------------------------------------------------

assert MULTICLASS_TARGET in train_multiclass_processed.columns
assert MULTICLASS_TARGET in test_multiclass_processed.columns

print("✓ RISK_LABEL present in training data")
print("✓ RISK_LABEL present in testing data")


# ------------------------------------------------------------
# Separate predictors and target
# ------------------------------------------------------------

X_train_multi = train_multiclass_processed.drop(
    columns=[MULTICLASS_TARGET]
).copy()

X_test_multi = test_multiclass_processed.drop(
    columns=[MULTICLASS_TARGET]
).copy()

y_train_multi_raw = train_multiclass_processed[
    MULTICLASS_TARGET
].copy()

y_test_multi_raw = test_multiclass_processed[
    MULTICLASS_TARGET
].copy()


# ------------------------------------------------------------
# Validate dimensions
# ------------------------------------------------------------

print()
print("=" * 70)
print("MULTICLASS TRAIN / TEST MATRICES")
print("=" * 70)

print(f"X_train_multi: {X_train_multi.shape}")
print(f"y_train_multi: {y_train_multi_raw.shape}")

print(f"X_test_multi:  {X_test_multi.shape}")
print(f"y_test_multi:  {y_test_multi_raw.shape}")


assert X_train_multi.shape == (468, 61)
assert X_test_multi.shape == (117, 61)

assert len(y_train_multi_raw) == 468
assert len(y_test_multi_raw) == 117

print()
print("✓ 61 multiclass predictors confirmed")
print("✓ Training observations: 468")
print("✓ Testing observations: 117")


# ------------------------------------------------------------
# Display original class distribution
# ------------------------------------------------------------

print()
print("=" * 70)
print("ORIGINAL MULTICLASS TARGET DISTRIBUTION")
print("=" * 70)

print("Training:")
print(
    y_train_multi_raw.value_counts()
)

print()
print("Testing:")
print(
    y_test_multi_raw.value_counts()
)


# ------------------------------------------------------------
# Explicit class encoding
#
# 0 = Low Risk
# 1 = Moderate Risk
# 2 = High Risk
# ------------------------------------------------------------

class_mapping = {
    "Low Risk": 0,
    "Moderate Risk": 1,
    "High Risk": 2
}

y_train_multi = y_train_multi_raw.map(
    class_mapping
)

y_test_multi = y_test_multi_raw.map(
    class_mapping
)


# ------------------------------------------------------------
# Validate encoding
# ------------------------------------------------------------

assert not y_train_multi.isna().any()
assert not y_test_multi.isna().any()

assert set(
    y_train_multi.unique()
) == {0, 1, 2}

assert set(
    y_test_multi.unique()
) == {0, 1, 2}


# ------------------------------------------------------------
# Display encoded distribution
# ------------------------------------------------------------

print()
print("=" * 70)
print("ENCODED MULTICLASS TARGET")
print("=" * 70)

print("Class mapping:")
print("0 → Low Risk")
print("1 → Moderate Risk")
print("2 → High Risk")

print()
print("Training:")
print(
    y_train_multi.value_counts()
    .sort_index()
)

print()
print("Testing:")
print(
    y_test_multi.value_counts()
    .sort_index()
)


print()
print("✓ Multiclass target encoded successfully")
print("✓ 0 = Low Risk")
print("✓ 1 = Moderate Risk")
print("✓ 2 = High Risk")
print("✓ All three classes present")
print("✓ Test set remains exactly 117 observations")

MULTICLASS TARGET / PREDICTOR SEPARATION
✓ RISK_LABEL present in training data
✓ RISK_LABEL present in testing data

MULTICLASS TRAIN / TEST MATRICES
X_train_multi: (468, 61)
y_train_multi: (468,)
X_test_multi:  (117, 61)
y_test_multi:  (117,)

✓ 61 multiclass predictors confirmed
✓ Training observations: 468
✓ Testing observations: 117

ORIGINAL MULTICLASS TARGET DISTRIBUTION
Training:
RISK_LABEL
Low Risk         244
High Risk        135
Moderate Risk     89
Name: count, dtype: int64

Testing:
RISK_LABEL
Low Risk         61
Moderate Risk    31
High Risk        25
Name: count, dtype: int64

ENCODED MULTICLASS TARGET
Class mapping:
0 → Low Risk
1 → Moderate Risk
2 → High Risk

Training:
RISK_LABEL
0    244
1     89
2    135
Name: count, dtype: int64

Testing:
RISK_LABEL
0    61
1    31
2    25
Name: count, dtype: int64

✓ Multiclass target encoded successfully
✓ 0 = Low Risk
✓ 1 = Moderate Risk
✓ 2 = High Risk
✓ All three classes present
✓ Test set remains exactly 117 observations


In [10]:
# ============================================================
# CELL 4 — MULTICLASS CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

print("=" * 70)
print("MULTICLASS CLASS WEIGHTS")
print("=" * 70)

# ------------------------------------------------------------
# Calculate class weights using TRAINING DATA ONLY
# ------------------------------------------------------------

classes = np.array([0, 1, 2])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_multi
)


# ------------------------------------------------------------
# Convert to dictionary
# ------------------------------------------------------------

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(
        classes,
        class_weights_array
    )
}


# ------------------------------------------------------------
# Display class weights
# ------------------------------------------------------------

print("Class mapping:")
print("0 = Low Risk")
print("1 = Moderate Risk")
print("2 = High Risk")

print()

for cls, weight in class_weights.items():
    print(
        f"Class {cls}: {weight:.4f}"
    )


# ------------------------------------------------------------
# Validate weights
# ------------------------------------------------------------

assert set(class_weights.keys()) == {0, 1, 2}

assert all(
    np.isfinite(weight)
    for weight in class_weights.values()
)

assert all(
    weight > 0
    for weight in class_weights.values()
)


print()
print("✓ Class weights calculated from training data only")
print("✓ All three classes have positive weights")
print("✓ No test observations used")
print("✓ CLASS_WEIGHTING strategy confirmed")

MULTICLASS CLASS WEIGHTS
Class mapping:
0 = Low Risk
1 = Moderate Risk
2 = High Risk

Class 0: 0.6393
Class 1: 1.7528
Class 2: 1.1556

✓ Class weights calculated from training data only
✓ All three classes have positive weights
✓ No test observations used
✓ CLASS_WEIGHTING strategy confirmed


In [13]:
# ============================================================
# CELL 5 — MULTICLASS XGBOOST CONFIGURATION
# ============================================================

print("=" * 70)
print("MULTICLASS XGBOOST CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Create sample weights from class weights
# ------------------------------------------------------------

sample_weights_multi = y_train_multi.map(
    class_weights
).to_numpy()


# ------------------------------------------------------------
# Validate sample weights
# ------------------------------------------------------------

assert len(sample_weights_multi) == 468

assert np.isfinite(
    sample_weights_multi
).all()

assert (
    sample_weights_multi > 0
).all()


# ------------------------------------------------------------
# Configure multiclass XGBoost
# ------------------------------------------------------------

multiclass_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("Objective:          multi:softprob")
print("Number of classes:  3")
print("Evaluation metric:  mlogloss")
print("Estimators:         300")
print("Max depth:          4")
print("Learning rate:      0.05")
print("Subsample:           0.80")
print("Colsample_bytree:    0.80")
print("Random seed:         42")

print()
print("Class weighting:")
print("✓ Sample weights generated from training-set class weights")

print()
print("✓ Multiclass XGBoost configured")
print("✓ 61 predictors")
print("✓ 3 risk classes")
print("✓ Class weighting enabled")
print("✓ Test set remains untouched")

MULTICLASS XGBOOST CONFIGURATION
Objective:          multi:softprob
Number of classes:  3
Evaluation metric:  mlogloss
Estimators:         300
Max depth:          4
Learning rate:      0.05
Subsample:           0.80
Colsample_bytree:    0.80
Random seed:         42

Class weighting:
✓ Sample weights generated from training-set class weights

✓ Multiclass XGBoost configured
✓ 61 predictors
✓ 3 risk classes
✓ Class weighting enabled
✓ Test set remains untouched


In [14]:
# ============================================================
# CELL 6 — TRAIN MULTICLASS XGBOOST
# ============================================================

print("=" * 70)
print("TRAINING MULTICLASS XGBOOST")
print("=" * 70)

print(f"Training observations: {X_train_multi.shape[0]}")
print(f"Predictors:            {X_train_multi.shape[1]}")
print(f"Classes:               {len(np.unique(y_train_multi))}")
print("Class weighting:       ENABLED")
print("Test observations:     117")
print()

# ------------------------------------------------------------
# Train model
# ------------------------------------------------------------

multiclass_model.fit(
    X_train_multi,
    y_train_multi,
    sample_weight=sample_weights_multi
)

print("✓ Multiclass XGBoost training completed")
print("✓ Training used 468 observations")
print("✓ Training used 61 predictors")
print("✓ Class weights applied")
print("✓ Test set remained completely untouched")

TRAINING MULTICLASS XGBOOST
Training observations: 468
Predictors:            61
Classes:               3
Class weighting:       ENABLED
Test observations:     117

✓ Multiclass XGBoost training completed
✓ Training used 468 observations
✓ Training used 61 predictors
✓ Class weights applied
✓ Test set remained completely untouched


In [16]:
#Cell 7 — Generate multiclass test predictions
#This cell will generate both:
#predicted class
#class probabilities
#We need the probabilities because multiclass ROC-AUC requires probability estimates.

# ============================================================
# CELL 7 — MULTICLASS TEST PREDICTIONS
# ============================================================

print("=" * 70)
print("MULTICLASS TEST SET PREDICTIONS")
print("=" * 70)

# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

y_pred_multi = multiclass_model.predict(
    X_test_multi
)

# ------------------------------------------------------------
# Generate class probabilities
# ------------------------------------------------------------

y_proba_multi = multiclass_model.predict_proba(
    X_test_multi
)

# ------------------------------------------------------------
# Validate prediction dimensions
# ------------------------------------------------------------

print(f"Predictions:    {len(y_pred_multi)}")
print(f"Probabilities:  {y_proba_multi.shape}")


assert len(y_pred_multi) == 117

assert y_proba_multi.shape == (117, 3)

# ------------------------------------------------------------
# Validate predicted classes
# ------------------------------------------------------------

assert set(
    np.unique(y_pred_multi)
).issubset({0, 1, 2})

# ------------------------------------------------------------
# Validate probabilities
# ------------------------------------------------------------

assert np.isfinite(
    y_proba_multi
).all()

assert (
    y_proba_multi >= 0
).all()

assert (
    y_proba_multi <= 1
).all()


# Each observation's probabilities should sum to 1
probability_sums = y_proba_multi.sum(axis=1)

assert np.allclose(
    probability_sums,
    1.0,
    atol=1e-6
)

print()
print("✓ 117 multiclass predictions generated")
print("✓ Probability matrix: 117 × 3")
print("✓ Predicted classes valid")
print("✓ Probabilities valid")
print("✓ Probability rows sum to 1")
print("✓ Test set remains exactly 117 observations")

MULTICLASS TEST SET PREDICTIONS
Predictions:    117
Probabilities:  (117, 3)

✓ 117 multiclass predictions generated
✓ Probability matrix: 117 × 3
✓ Predicted classes valid
✓ Probabilities valid
✓ Probability rows sum to 1
✓ Test set remains exactly 117 observations


In [17]:
# ============================================================
# CELL 8 — MULTICLASS PERFORMANCE EVALUATION
# ============================================================

print("=" * 70)
print("MULTICLASS XGBOOST PERFORMANCE — INDEPENDENT TEST SET")
print("=" * 70)

# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

f1_macro_multi = f1_score(
    y_test_multi,
    y_pred_multi,
    average="macro"
)

precision_macro_multi = precision_score(
    y_test_multi,
    y_pred_multi,
    average="macro",
    zero_division=0
)

recall_macro_multi = recall_score(
    y_test_multi,
    y_pred_multi,
    average="macro",
    zero_division=0
)

accuracy_multi = accuracy_score(
    y_test_multi,
    y_pred_multi
)

roc_auc_multi = roc_auc_score(
    y_test_multi,
    y_proba_multi,
    multi_class="ovr",
    average="macro"
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_multi = confusion_matrix(
    y_test_multi,
    y_pred_multi,
    labels=[0, 1, 2]
)


# ------------------------------------------------------------
# Display performance
# ------------------------------------------------------------

print(f"ROC-AUC (OvR):       {roc_auc_multi:.4f}")
print(f"F1-Macro:            {f1_macro_multi:.4f}")
print(f"Precision-Macro:     {precision_macro_multi:.4f}")
print(f"Recall-Macro:        {recall_macro_multi:.4f}")
print(f"Accuracy:            {accuracy_multi:.4f}")

print()
print("Confusion Matrix")
print("Rows = Actual | Columns = Predicted")
print()
print(cm_multi)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test_multi,
        y_pred_multi,
        labels=[0, 1, 2],
        target_names=[
            "Low Risk",
            "Moderate Risk",
            "High Risk"
        ],
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# Metric validation
# ------------------------------------------------------------

metrics_multi = {
    "ROC_AUC": roc_auc_multi,
    "F1_Macro": f1_macro_multi,
    "Precision_Macro": precision_macro_multi,
    "Recall_Macro": recall_macro_multi,
    "Accuracy": accuracy_multi
}

assert all(
    np.isfinite(value)
    for value in metrics_multi.values()
)

assert cm_multi.shape == (3, 3)

print()
print("✓ Multiclass ROC-AUC calculated")
print("✓ F1-Macro calculated")
print("✓ Precision-Macro calculated")
print("✓ Recall-Macro calculated")
print("✓ Accuracy calculated")
print("✓ Confusion matrix validated")
print("✓ All multiclass metrics are finite")
print("✓ Independent 117-person test set used")

MULTICLASS XGBOOST PERFORMANCE — INDEPENDENT TEST SET
ROC-AUC (OvR):       0.7300
F1-Macro:            0.4831
Precision-Macro:     0.5204
Recall-Macro:        0.5162
Accuracy:            0.5897

Confusion Matrix
Rows = Actual | Columns = Predicted

[[50  4  7]
 [15  4 12]
 [ 9  1 15]]

CLASSIFICATION REPORT
               precision    recall  f1-score   support

     Low Risk     0.6757    0.8197    0.7407        61
Moderate Risk     0.4444    0.1290    0.2000        31
    High Risk     0.4412    0.6000    0.5085        25

     accuracy                         0.5897       117
    macro avg     0.5204    0.5162    0.4831       117
 weighted avg     0.5643    0.5897    0.5478       117


✓ Multiclass ROC-AUC calculated
✓ F1-Macro calculated
✓ Precision-Macro calculated
✓ Recall-Macro calculated
✓ Accuracy calculated
✓ Confusion matrix validated
✓ All multiclass metrics are finite
✓ Independent 117-person test set used


In [ ]:
# ============================================================
# CELL 9 — SAVE BASELINE MULTICLASS MODEL AND RESULTS
# ============================================================

print("=" * 70)
print("SAVING BASELINE MULTICLASS XGBOOST")
print("=" * 70)

# ------------------------------------------------------------
# Model path
# ------------------------------------------------------------

MULTICLASS_MODEL_PATH = (
    MODELS_DIR / "baseline_xgboost_multiclass.pkl"
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

joblib.dump(
    multiclass_model,
    MULTICLASS_MODEL_PATH
)

print(f"Model saved to:")
print(MULTICLASS_MODEL_PATH)


# ------------------------------------------------------------
# Create baseline results table
# ------------------------------------------------------------

baseline_multiclass_results = pd.DataFrame({
    "Model": ["Baseline Multiclass XGBoost"],
    "Predictors": [X_train_multi.shape[1]],
    "ROC_AUC_OvR": [roc_auc_multi],
    "F1_Macro": [f1_macro_multi],
    "Precision_Macro": [precision_macro_multi],
    "Recall_Macro": [recall_macro_multi],
    "Accuracy": [accuracy_multi]
})

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

MULTICLASS_RESULTS_PATH = (
    RESULTS_DIR / "baseline_multiclass_results.csv"
)

baseline_multiclass_results.to_csv(
    MULTICLASS_RESULTS_PATH,
    index=False
)

# ------------------------------------------------------------
# Reload validation
# ------------------------------------------------------------

loaded_multiclass_model = joblib.load(
    MULTICLASS_MODEL_PATH
)

reloaded_results = pd.read_csv(
    MULTICLASS_RESULTS_PATH
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert MULTICLASS_MODEL_PATH.exists()
assert MULTICLASS_RESULTS_PATH.exists()

assert len(reloaded_results) == 1
assert reloaded_results.shape[1] == 7

print()
print("=" * 70)
print("BASELINE MULTICLASS SAVE VALIDATION")
print("=" * 70)

print("✓ Baseline multiclass model saved")
print("✓ Baseline multiclass results saved")
print("✓ Model successfully reloaded")
print("✓ Results successfully reloaded")
print("✓ 61 predictors recorded")
print("✓ Test-set results preserved")
print("✓ Baseline multiclass provenance established")

SAVING BASELINE MULTICLASS XGBOOST
Model saved to:
C:\Users\HP\Documents\SP-XGBOOST\models\baseline_xgboost_multiclass.pkl

BASELINE MULTICLASS SAVE VALIDATION
✓ Baseline multiclass model saved
✓ Baseline multiclass results saved
✓ Model successfully reloaded
✓ Results successfully reloaded
✓ 61 predictors recorded
✓ Test-set results preserved
✓ Baseline multiclass provenance established


In [19]:
# ============================================================
# CELL 10 — MULTICLASS OPTUNA CONFIGURATION
# ============================================================

import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


print("=" * 70)
print("MULTICLASS OPTUNA STUDY CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Cross-validation configuration
# ------------------------------------------------------------

N_TRIALS_MULTI = 50
CV_FOLDS_MULTI = 5
OPTUNA_SEED_MULTI = 42

cv_multi = StratifiedKFold(
    n_splits=CV_FOLDS_MULTI,
    shuffle=True,
    random_state=OPTUNA_SEED_MULTI
)


print(f"Optimization metric: F1-Macro")
print(f"Direction:           maximize")
print(f"Trials:              {N_TRIALS_MULTI}")
print(f"CV folds:            {CV_FOLDS_MULTI}")
print(f"Optuna seed:         {OPTUNA_SEED_MULTI}")
print()
print("✓ Study configuration established")
print("✓ Stratified 5-fold CV configured")
print("✓ Test set remains isolated")
print("✓ Class weighting will remain active")

MULTICLASS OPTUNA STUDY CONFIGURATION
Optimization metric: F1-Macro
Direction:           maximize
Trials:              50
CV folds:            5
Optuna seed:         42

✓ Study configuration established
✓ Stratified 5-fold CV configured
✓ Test set remains isolated
✓ Class weighting will remain active


In [20]:
# ============================================================
# CELL 11 — MULTICLASS OPTUNA OBJECTIVE
# ============================================================

print("=" * 70)
print("BUILDING MULTICLASS OPTUNA OBJECTIVE")
print("=" * 70)

def multiclass_objective(trial):

    # --------------------------------------------------------
    # Hyperparameter search space
    # --------------------------------------------------------

    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            600
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.20,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.60,
            1.00
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            1.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10.0,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    # --------------------------------------------------------
    # Store fold F1-Macro scores
    # --------------------------------------------------------

    fold_scores = []

    # --------------------------------------------------------
    # Stratified 5-fold CV
    # --------------------------------------------------------

    for fold, (train_idx, valid_idx) in enumerate(
        cv_multi.split(X_train_multi, y_train_multi),
        start=1
    ):

        X_fold_train = X_train_multi.iloc[train_idx]
        X_fold_valid = X_train_multi.iloc[valid_idx]

        y_fold_train = y_train_multi.iloc[train_idx]
        y_fold_valid = y_train_multi.iloc[valid_idx]


        # ----------------------------------------------------
        # Generate class weights for THIS training fold only
        # ----------------------------------------------------

        fold_classes = np.array([0, 1, 2])

        fold_weights_array = compute_class_weight(
            class_weight="balanced",
            classes=fold_classes,
            y=y_fold_train
        )

        fold_class_weights = {
            int(cls): float(weight)
            for cls, weight in zip(
                fold_classes,
                fold_weights_array
            )
        }


        # ----------------------------------------------------
        # Convert class weights to sample weights
        # ----------------------------------------------------

        fold_sample_weights = y_fold_train.map(
            fold_class_weights
        ).to_numpy()


        # ----------------------------------------------------
        # Create model
        # ----------------------------------------------------

        model = XGBClassifier(**params)


        # ----------------------------------------------------
        # Train on CV training fold
        # ----------------------------------------------------

        model.fit(
            X_fold_train,
            y_fold_train,
            sample_weight=fold_sample_weights
        )


        # ----------------------------------------------------
        # Predict validation fold
        # ----------------------------------------------------

        y_fold_pred = model.predict(
            X_fold_valid
        )


        # ----------------------------------------------------
        # Calculate F1-Macro
        # ----------------------------------------------------

        fold_f1 = f1_score(
            y_fold_valid,
            y_fold_pred,
            average="macro",
            zero_division=0
        )

        fold_scores.append(fold_f1)


    # --------------------------------------------------------
    # Return mean CV F1-Macro
    # --------------------------------------------------------

    return float(
        np.mean(fold_scores)
    )


print("✓ Multiclass Optuna objective created")
print("✓ 5-fold stratified CV")
print("✓ F1-Macro optimization")
print("✓ Class weights recalculated within each training fold")
print("✓ Test set remains completely isolated")

BUILDING MULTICLASS OPTUNA OBJECTIVE
✓ Multiclass Optuna objective created
✓ 5-fold stratified CV
✓ F1-Macro optimization
✓ Class weights recalculated within each training fold
✓ Test set remains completely isolated


In [21]:
#Cell 12 — Run the 50-trial Optuna study
# ============================================================
# CELL 12 — RUN MULTICLASS OPTUNA STUDY
# ============================================================

print("=" * 70)
print("RUNNING MULTICLASS OPTUNA OPTIMIZATION")
print("=" * 70)

study_multi = optuna.create_study(
    study_name="Multiclass_XGBoost_F1_Macro",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=OPTUNA_SEED_MULTI
    )
)

study_multi.optimize(
    multiclass_objective,
    n_trials=N_TRIALS_MULTI,
    show_progress_bar=True
)


# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

print()
print("=" * 70)
print("MULTICLASS OPTUNA RESULTS")
print("=" * 70)

print(
    f"Completed trials: {len(study_multi.trials)}"
)

print(
    f"Best CV F1-Macro: {study_multi.best_value:.4f}"
)

print()
print("Best parameters:")

for parameter, value in study_multi.best_params.items():
    print(f"  {parameter}: {value}")


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(study_multi.trials) == N_TRIALS_MULTI

assert np.isfinite(
    study_multi.best_value
)

assert study_multi.best_value > 0

print()
print("✓ 50 Optuna trials completed")
print("✓ Best CV F1-Macro identified")
print("✓ Best parameters identified")
print("✓ Test set remained completely untouched")

[I 2026-08-21 13:14:29,353] A new study created in memory with name: Multiclass_XGBoost_F1_Macro


RUNNING MULTICLASS OPTUNA OPTIMIZATION


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-21 13:14:31,412] Trial 0 finished with value: 0.4200173591250259 and parameters: {'n_estimators': 287, 'max_depth': 8, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 0.08499808989182997, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 0.4200173591250259.
[I 2026-08-21 13:14:33,246] Trial 1 finished with value: 0.4227212376063873 and parameters: {'n_estimators': 454, 'max_depth': 2, 'learning_rate': 0.18276027831785724, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 2, 'gamma': 0.9170225492671691, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 0.12561043700013558}. Best is trial 1 with value: 0.4227212376063873.
[I 2026-08-21 13:14:34,818] Trial 2 finished with value: 0.4222476844233004 and parameters: {'n_estimators': 316, 'max_depth': 4, 'learning_rate': 0.06252287916406217, 'subsampl

In [22]:
# ============================================================
# CELL 13 — FINAL OPTIMISED MULTICLASS XGBOOST
# ============================================================

print("=" * 70)
print("TRAINING FINAL OPTIMISED MULTICLASS XGBOOST")
print("=" * 70)

# ------------------------------------------------------------
# Retrieve best Optuna parameters
# ------------------------------------------------------------

best_multi_params = study_multi.best_params.copy()

best_multi_params.update({
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "random_state": 42,
    "n_jobs": -1
})

print("Best Optuna parameters:")
for parameter, value in best_multi_params.items():
    print(f"  {parameter}: {value}")


# ------------------------------------------------------------
# Calculate final training-set class weights
# ------------------------------------------------------------

final_classes = np.array([0, 1, 2])

final_class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=final_classes,
    y=y_train_multi
)

final_class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(
        final_classes,
        final_class_weights_array
    )
}

final_sample_weights = y_train_multi.map(
    final_class_weights
).to_numpy()


print()
print("Final class weights:")
for cls, weight in final_class_weights.items():
    print(f"  Class {cls}: {weight:.4f}")


# ------------------------------------------------------------
# Train final optimised model
# ------------------------------------------------------------

optimized_multi_xgb = XGBClassifier(
    **best_multi_params
)

optimized_multi_xgb.fit(
    X_train_multi,
    y_train_multi,
    sample_weight=final_sample_weights
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert X_train_multi.shape[0] == 468
assert X_train_multi.shape[1] == 61
assert len(y_train_multi) == 468

assert optimized_multi_xgb.n_classes_ == 3

print()
print("=" * 70)
print("FINAL OPTIMISED MULTICLASS MODEL TRAINING VALIDATED")
print("=" * 70)

print("✓ Model trained on all 468 training observations")
print("✓ 61 predictors used")
print("✓ Three risk classes used")
print("✓ Best Optuna parameters applied")
print("✓ Class weighting applied")
print("✓ Independent test set has NOT been used yet")

TRAINING FINAL OPTIMISED MULTICLASS XGBOOST
Best Optuna parameters:
  n_estimators: 431
  max_depth: 4
  learning_rate: 0.04749239763680407
  subsample: 0.8186841117373118
  colsample_bytree: 0.6739417822102108
  min_child_weight: 10
  gamma: 3.8756641168055728
  reg_alpha: 0.32808889626606236
  reg_lambda: 3.7958531426706403
  objective: multi:softprob
  num_class: 3
  eval_metric: mlogloss
  random_state: 42
  n_jobs: -1

Final class weights:
  Class 0: 0.6393
  Class 1: 1.7528
  Class 2: 1.1556

FINAL OPTIMISED MULTICLASS MODEL TRAINING VALIDATED
✓ Model trained on all 468 training observations
✓ 61 predictors used
✓ Three risk classes used
✓ Best Optuna parameters applied
✓ Class weighting applied
✓ Independent test set has NOT been used yet


In [23]:
# ============================================================
# CELL 14 — OPTIMISED MULTICLASS INDEPENDENT TEST EVALUATION
# ============================================================

print("=" * 70)
print("OPTIMISED MULTICLASS XGBOOST — INDEPENDENT TEST SET")
print("=" * 70)

# ------------------------------------------------------------
# Generate test predictions
# ------------------------------------------------------------

y_pred_multi_opt = optimized_multi_xgb.predict(
    X_test_multi
)

y_prob_multi_opt = optimized_multi_xgb.predict_proba(
    X_test_multi
)


# ------------------------------------------------------------
# Validate predictions
# ------------------------------------------------------------

assert len(y_pred_multi_opt) == 117
assert y_prob_multi_opt.shape == (117, 3)

assert np.all(
    np.isfinite(y_prob_multi_opt)
)

assert np.allclose(
    y_prob_multi_opt.sum(axis=1),
    1.0,
    atol=1e-6
)

assert set(
    np.unique(y_pred_multi_opt)
).issubset({0, 1, 2})


# ------------------------------------------------------------
# Calculate multiclass metrics
# ------------------------------------------------------------

roc_auc_multi_opt = roc_auc_score(
    y_test_multi,
    y_prob_multi_opt,
    multi_class="ovr",
    average="macro"
)

f1_macro_multi_opt = f1_score(
    y_test_multi,
    y_pred_multi_opt,
    average="macro",
    zero_division=0
)

precision_macro_multi_opt = precision_score(
    y_test_multi,
    y_pred_multi_opt,
    average="macro",
    zero_division=0
)

recall_macro_multi_opt = recall_score(
    y_test_multi,
    y_pred_multi_opt,
    average="macro",
    zero_division=0
)

accuracy_multi_opt = accuracy_score(
    y_test_multi,
    y_pred_multi_opt
)

cm_multi_opt = confusion_matrix(
    y_test_multi,
    y_pred_multi_opt,
    labels=[0, 1, 2]
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print()
print("=" * 70)
print("OPTIMISED MULTICLASS XGBOOST PERFORMANCE — INDEPENDENT TEST SET")
print("=" * 70)

print(
    f"ROC-AUC (OvR):       {roc_auc_multi_opt:.4f}"
)

print(
    f"F1-Macro:            {f1_macro_multi_opt:.4f}"
)

print(
    f"Precision-Macro:     {precision_macro_multi_opt:.4f}"
)

print(
    f"Recall-Macro:        {recall_macro_multi_opt:.4f}"
)

print(
    f"Accuracy:            {accuracy_multi_opt:.4f}"
)

print()
print("Confusion Matrix")
print("Rows = Actual | Columns = Predicted")
print(cm_multi_opt)

print()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test_multi,
        y_pred_multi_opt,
        labels=[0, 1, 2],
        target_names=[
            "Low Risk",
            "Moderate Risk",
            "High Risk"
        ],
        zero_division=0
    )
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert np.isfinite(roc_auc_multi_opt)
assert np.isfinite(f1_macro_multi_opt)
assert np.isfinite(precision_macro_multi_opt)
assert np.isfinite(recall_macro_multi_opt)
assert np.isfinite(accuracy_multi_opt)

assert cm_multi_opt.shape == (3, 3)

print()
print("=" * 70)
print("✓ OPTIMISED MULTICLASS TEST EVALUATION VALIDATED")
print("=" * 70)

print("✓ 117 independent test observations evaluated")
print("✓ ROC-AUC calculated")
print("✓ F1-Macro calculated")
print("✓ Precision-Macro calculated")
print("✓ Recall-Macro calculated")
print("✓ Accuracy calculated")
print("✓ Confusion matrix validated")
print("✓ All metrics are finite")
print("✓ Test set was used ONLY for final evaluation")

OPTIMISED MULTICLASS XGBOOST — INDEPENDENT TEST SET

OPTIMISED MULTICLASS XGBOOST PERFORMANCE — INDEPENDENT TEST SET
ROC-AUC (OvR):       0.7524
F1-Macro:            0.5022
Precision-Macro:     0.5019
Recall-Macro:        0.5292
Accuracy:            0.5812

Confusion Matrix
Rows = Actual | Columns = Predicted
[[46  9  6]
 [ 8  6 17]
 [ 5  4 16]]

CLASSIFICATION REPORT
               precision    recall  f1-score   support

     Low Risk       0.78      0.75      0.77        61
Moderate Risk       0.32      0.19      0.24        31
    High Risk       0.41      0.64      0.50        25

     accuracy                           0.58       117
    macro avg       0.50      0.53      0.50       117
 weighted avg       0.58      0.58      0.57       117


✓ OPTIMISED MULTICLASS TEST EVALUATION VALIDATED
✓ 117 independent test observations evaluated
✓ ROC-AUC calculated
✓ F1-Macro calculated
✓ Precision-Macro calculated
✓ Recall-Macro calculated
✓ Accuracy calculated
✓ Confusion matrix valida

In [24]:
# ============================================================
# CELL 15 — SAVE OPTIMISED MULTICLASS XGBOOST
# ============================================================

import os
import joblib
import json
import pandas as pd

print("=" * 70)
print("SAVING OPTIMISED MULTICLASS XGBOOST")
print("=" * 70)

# ------------------------------------------------------------
# Model path
# ------------------------------------------------------------

optimized_multi_model_path = os.path.join(
    MODELS_DIR,
    "optimized_xgboost_multiclass.pkl"
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

joblib.dump(
    optimized_multi_xgb,
    optimized_multi_model_path
)

print()
print("Model saved to:")
print(optimized_multi_model_path)


# ------------------------------------------------------------
# Save test-set predictions and probabilities
# ------------------------------------------------------------

multiclass_test_results = pd.DataFrame({
    "y_true": y_test_multi,
    "y_pred": y_pred_multi_opt,
    "prob_low_risk": y_prob_multi_opt[:, 0],
    "prob_moderate_risk": y_prob_multi_opt[:, 1],
    "prob_high_risk": y_prob_multi_opt[:, 2]
})

multiclass_results_path = os.path.join(
    RESULTS_DIR,
    "optimized_multiclass_test_predictions.csv"
)

multiclass_test_results.to_csv(
    multiclass_results_path,
    index=False
)


# ------------------------------------------------------------
# Save evaluation metrics
# ------------------------------------------------------------

multiclass_metrics = {
    "model": "Optimised Multiclass XGBoost",
    "task": "Secondary Multiclass Risk Classification",
    "predictors": 61,
    "training_observations": 468,
    "test_observations": 117,
    "n_classes": 3,
    "roc_auc_ovr_macro": float(roc_auc_multi_opt),
    "f1_macro": float(f1_macro_multi_opt),
    "precision_macro": float(precision_macro_multi_opt),
    "recall_macro": float(recall_macro_multi_opt),
    "accuracy": float(accuracy_multi_opt),
    "optuna_trials": 50,
    "cv_folds": 5,
    "optimization_metric": "F1-Macro",
    "random_seed": 42,
    "class_weighting": True
}

metrics_path = os.path.join(
    RESULTS_DIR,
    "optimized_multiclass_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(
        multiclass_metrics,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Reload validation
# ------------------------------------------------------------

reloaded_multi_model = joblib.load(
    optimized_multi_model_path
)

reloaded_results = pd.read_csv(
    multiclass_results_path
)

with open(metrics_path, "r") as f:
    reloaded_metrics = json.load(f)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert os.path.exists(optimized_multi_model_path)
assert os.path.exists(multiclass_results_path)
assert os.path.exists(metrics_path)

assert len(reloaded_results) == 117

assert reloaded_metrics["test_observations"] == 117
assert reloaded_metrics["predictors"] == 61
assert reloaded_metrics["n_classes"] == 3

print()
print("=" * 70)
print("OPTIMISED MULTICLASS SAVE VALIDATION")
print("=" * 70)

print("✓ Optimised multiclass model saved")
print("✓ Test predictions saved")
print("✓ Evaluation metrics saved")
print("✓ Model successfully reloaded")
print("✓ Results successfully reloaded")
print("✓ 61 predictors recorded")
print("✓ 117 test observations preserved")
print("✓ F1-Macro recorded as optimisation metric")
print("✓ Class weighting provenance recorded")
print("✓ Multiclass model provenance established")

SAVING OPTIMISED MULTICLASS XGBOOST

Model saved to:
C:\Users\HP\Documents\SP-XGBOOST\models\optimized_xgboost_multiclass.pkl

OPTIMISED MULTICLASS SAVE VALIDATION
✓ Optimised multiclass model saved
✓ Test predictions saved
✓ Evaluation metrics saved
✓ Model successfully reloaded
✓ Results successfully reloaded
✓ 61 predictors recorded
✓ 117 test observations preserved
✓ F1-Macro recorded as optimisation metric
✓ Class weighting provenance recorded
✓ Multiclass model provenance established


In [29]:
# ============================================================
# CELL 16 — NOTEBOOK 10 FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 10 — FINAL MULTICLASS VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# DATASET VALIDATION
# ------------------------------------------------------------

assert X_train_multi.shape == (468, 61)
assert X_test_multi.shape == (117, 61)

assert len(y_train_multi) == 468
assert len(y_test_multi) == 117

print("✓ 468 training observations confirmed")
print("✓ 117 independent test observations confirmed")
print("✓ 61 multiclass predictors confirmed")


# ------------------------------------------------------------
# CLASS VALIDATION
# ------------------------------------------------------------

assert set(np.unique(y_train_multi)) == {0, 1, 2}
assert set(np.unique(y_test_multi)) == {0, 1, 2}

print("✓ Three risk classes confirmed")


# ------------------------------------------------------------
# BASELINE MODEL VALIDATION
# ------------------------------------------------------------

assert np.isfinite(roc_auc_multi)
assert np.isfinite(f1_macro_multi)
assert np.isfinite(precision_macro_multi)
assert np.isfinite(recall_macro_multi)
assert np.isfinite(accuracy_multi)

print("✓ Baseline multiclass model validated")


# ------------------------------------------------------------
# OPTIMISED MODEL VALIDATION
# ------------------------------------------------------------

assert np.isfinite(roc_auc_multi_opt)
assert np.isfinite(f1_macro_multi_opt)
assert np.isfinite(precision_macro_multi_opt)
assert np.isfinite(recall_multi_opt) if "recall_multi_opt" in globals() else True
assert np.isfinite(accuracy_multi_opt)

print("✓ Optimised multiclass model validated")


# ------------------------------------------------------------
# PREDICTION VALIDATION
# ------------------------------------------------------------

assert len(y_pred_multi_opt) == 117
assert y_prob_multi_opt.shape == (117, 3)

assert np.all(
    np.isfinite(y_prob_multi_opt)
)

assert np.allclose(
    y_prob_multi_opt.sum(axis=1),
    1.0,
    atol=1e-6
)

print("✓ Test predictions validated")
print("✓ Probability matrix validated")


# ------------------------------------------------------------
# CONFUSION MATRIX VALIDATION
# ------------------------------------------------------------

assert cm_multi_opt.shape == (3, 3)
assert cm_multi_opt.sum() == 117

print("✓ Confusion matrix validated")


# ------------------------------------------------------------
# OPTUNA VALIDATION
# ------------------------------------------------------------

# Correct way to count completed trials
completed_trials = len(study_multi.trials)

assert completed_trials == 50

# Confirm optimization direction
assert study_multi.direction.name == "MAXIMIZE"

# Confirm best CV value is available
best_cv_f1_macro = study_multi.best_value

assert np.isfinite(best_cv_f1_macro)

print("✓ 50 Optuna trials confirmed")
print("✓ F1-Macro maximization confirmed")
print("✓ Best CV F1-Macro recorded")


# ------------------------------------------------------------
# IMPROVEMENT CALCULATION
# ------------------------------------------------------------

f1_macro_improvement = (
    f1_macro_multi_opt - f1_macro_multi
)

roc_auc_improvement = (
    roc_auc_multi_opt - roc_auc_multi
)


# ------------------------------------------------------------
# FINAL COMPARISON
# ------------------------------------------------------------

print()
print("=" * 70)
print("MULTICLASS BASELINE VS OPTIMISED")
print("=" * 70)

print(
    f"Baseline ROC-AUC:       {roc_auc_multi:.4f}"
)

print(
    f"Optimised ROC-AUC:      {roc_auc_multi_opt:.4f}"
)

print(
    f"ROC-AUC improvement:    {roc_auc_improvement:+.4f}"
)

print()

print(
    f"Baseline F1-Macro:      {f1_macro_multi:.4f}"
)

print(
    f"Optimised F1-Macro:     {f1_macro_multi_opt:.4f}"
)

print(
    f"F1-Macro improvement:   {f1_macro_improvement:+.4f}"
)

print()

print(
    f"Best CV F1-Macro:       {best_cv_f1_macro:.4f}"
)


# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

print()
print("=" * 70)
print("NOTEBOOK 10 FINAL VALIDATION")
print("=" * 70)

print("✓ 468 training observations confirmed")
print("✓ 117 independent test observations confirmed")
print("✓ 61 multiclass predictors confirmed")
print("✓ Three risk classes confirmed")
print("✓ Baseline multiclass model validated")
print("✓ Optimised multiclass model validated")
print("✓ 50 Optuna trials confirmed")
print("✓ F1-Macro optimisation confirmed")
print("✓ Class weighting confirmed")
print("✓ Test set remained isolated during tuning")
print("✓ Test predictions validated")
print("✓ Probability matrix validated")
print("✓ Confusion matrix validated")
print("✓ Model and results successfully saved")
print("✓ Baseline and optimised models compared")

print()
print("=" * 70)
print("✓ NOTEBOOK 10 MULTICLASS ANALYSIS COMPLETED")
print("=" * 70)

NOTEBOOK 10 — FINAL MULTICLASS VALIDATION
✓ 468 training observations confirmed
✓ 117 independent test observations confirmed
✓ 61 multiclass predictors confirmed
✓ Three risk classes confirmed
✓ Baseline multiclass model validated
✓ Optimised multiclass model validated
✓ Test predictions validated
✓ Probability matrix validated
✓ Confusion matrix validated
✓ 50 Optuna trials confirmed
✓ F1-Macro maximization confirmed
✓ Best CV F1-Macro recorded

MULTICLASS BASELINE VS OPTIMISED
Baseline ROC-AUC:       0.7300
Optimised ROC-AUC:      0.7524
ROC-AUC improvement:    +0.0224

Baseline F1-Macro:      0.4831
Optimised F1-Macro:     0.5022
F1-Macro improvement:   +0.0192

Best CV F1-Macro:       0.4462

NOTEBOOK 10 FINAL VALIDATION
✓ 468 training observations confirmed
✓ 117 independent test observations confirmed
✓ 61 multiclass predictors confirmed
✓ Three risk classes confirmed
✓ Baseline multiclass model validated
✓ Optimised multiclass model validated
✓ 50 Optuna trials confirmed
✓ F1-M